# Polar AccessLink OAuth Workflow

## Overview
This notebook demonstrates a complete end-to-end workflow for interacting with the Polar AccessLink API using OAuth 2.0 authorization code flow.

## Security Warnings ⚠️
- **NEVER hard-code secrets** in notebooks or commit them to version control
- **Use environment variables** for sensitive credentials
- **Rotate any exposed secrets** immediately if accidentally committed
- The `tokens_polar.json` file is gitignored - do NOT commit it

## Setup Requirements

### Environment Variables
Set the following environment variables before running this notebook:

```bash
export POLAR_CLIENT_ID="your_client_id_here"
export POLAR_CLIENT_SECRET="your_client_secret_here"
export POLAR_REDIRECT_PORT="5000"  # Default: 5000
export POLAR_MEMBER_ID="your_member_id_here"  # Optional, for registration
```

### Using .env File (Recommended)
Create a `.env` file in the project root (this file is gitignored):

```
POLAR_CLIENT_ID=your_client_id_here
POLAR_CLIENT_SECRET=your_client_secret_here
POLAR_REDIRECT_PORT=5000
POLAR_MEMBER_ID=your_member_id_here
```

Then load it using `python-dotenv`:
```python
from dotenv import load_dotenv
load_dotenv()
```

### Polar Developer Account
1. Register your application at https://admin.polaraccesslink.com/
2. Configure redirect URI: `http://localhost:5000/callback` (or your chosen port)
3. Note your Client ID and Client Secret

## Workflow Steps
1. **Task 1**: Configuration and helper functions
2. **Task 2**: Local callback server to capture authorization code
3. **Task 1b**: Exchange authorization code for access tokens
4. **Task 3**: Register user and access exercise data
5. **Task 4**: Testing and validation


## Task 1: Configuration and Helper Functions

In [18]:
import os
import json
import base64
import requests
from pathlib import Path

# Optional: Load from .env file if python-dotenv is available
try:
    from dotenv import load_dotenv
    load_dotenv()
    print("✓ Loaded environment from .env file")
except ImportError:
    print("ℹ python-dotenv not installed, using system environment variables")

# Load configuration from environment variables
CLIENT_ID = os.getenv('POLAR_CLIENT_ID')
CLIENT_SECRET = os.getenv('POLAR_CLIENT_SECRET')
REDIRECT_PORT = int(os.getenv('POLAR_REDIRECT_PORT', '5000'))
MEMBER_ID = os.getenv('POLAR_MEMBER_ID')

# Validate required environment variables
missing_vars = []
if not CLIENT_ID:
    missing_vars.append('POLAR_CLIENT_ID')
if not CLIENT_SECRET:
    missing_vars.append('POLAR_CLIENT_SECRET')

if missing_vars:
    raise ValueError(f"Missing required environment variables: {', '.join(missing_vars)}")

print(f"✓ Configuration loaded")
print(f"  - Client ID: {CLIENT_ID[:8]}...")
print(f"  - Redirect Port: {REDIRECT_PORT}")
print(f"  - Member ID: {MEMBER_ID if MEMBER_ID else 'Not set (will be obtained)'}")

# API endpoints
AUTH_URL = "https://flow.polar.com/oauth2/authorization"
TOKEN_URL = "https://polarremote.com/v2/oauth2/token"
API_BASE = "https://www.polaraccesslink.com/v3"

# Token storage file
TOKENS_FILE = Path("tokens_polar.json")

✓ Loaded environment from .env file
✓ Configuration loaded
  - Client ID: fddbcde4...
  - Redirect Port: 5000
  - Member ID: 61732059


In [19]:
# Helper Functions

def save_tokens(access_token, refresh_token=None, token_type="Bearer"):
    """Save tokens to JSON file (gitignored)."""
    tokens = {
        "access_token": access_token,
        "refresh_token": refresh_token,
        "token_type": token_type
    }
    with open(TOKENS_FILE, 'w') as f:
        json.dump(tokens, f, indent=2)
    print(f"✓ Tokens saved to {TOKENS_FILE}")


def load_tokens():
    """Load tokens from JSON file."""
    if not TOKENS_FILE.exists():
        return None
    with open(TOKENS_FILE, 'r') as f:
        tokens = json.load(f)
    return tokens


def encode_credentials(client_id, client_secret):
    """Encode credentials in Base64 for Basic Auth."""
    credentials = f"{client_id}:{client_secret}"
    encoded = base64.b64encode(credentials.encode('utf-8')).decode('utf-8')
    return encoded


def exchange_code_for_token(auth_code, redirect_uri):
    """
    Exchange authorization code for access and refresh tokens.
    CRITICAL: redirect_uri must EXACTLY match what was used in authorization request.
    """
    print(f"Exchanging authorization code for tokens...")
    print(f"  Using redirect_uri: {redirect_uri}")
    
    headers = {
        "Content-Type": "application/x-www-form-urlencoded",
        "Authorization": f"Basic {encode_credentials(CLIENT_ID, CLIENT_SECRET)}"
    }
    
    data = {
        "grant_type": "authorization_code",
        "code": auth_code,
        "redirect_uri": redirect_uri  # MUST match authorization request
    }
    
    response = requests.post(TOKEN_URL, headers=headers, data=data)
    
    if response.status_code != 200:
        print(f"❌ Token exchange failed: {response.status_code}")
        print(f"   Response: {response.text}")
        raise Exception(f"Token exchange failed: {response.text}")
    
    token_data = response.json()
    print("✓ Token exchange successful")
    return token_data


def refresh_access_token(refresh_token):
    """Refresh access token using refresh token."""
    print("Refreshing access token...")
    
    headers = {
        "Content-Type": "application/x-www-form-urlencoded",
        "Authorization": f"Basic {encode_credentials(CLIENT_ID, CLIENT_SECRET)}"
    }
    
    data = {
        "grant_type": "refresh_token",
        "refresh_token": refresh_token
    }
    
    response = requests.post(TOKEN_URL, headers=headers, data=data)
    
    if response.status_code != 200:
        print(f"❌ Token refresh failed: {response.status_code}")
        print(f"   Response: {response.text}")
        raise Exception(f"Token refresh failed: {response.text}")
    
    token_data = response.json()
    print("✓ Token refresh successful")
    return token_data


def ensure_token():
    """Placeholder for token management - load existing or prompt for new authorization."""
    tokens = load_tokens()
    if tokens:
        print(f"✓ Using existing tokens from {TOKENS_FILE}")
        return tokens['access_token']
    else:
        print("⚠ No tokens found. Please complete authorization flow first.")
        return None


def get_user_info(member_or_user_id, access_token):
    """Fetch user info from Polar API to get polar-user-id."""
    print(f"Fetching user info for ID: {member_or_user_id}...")
    
    headers = {
        "Authorization": f"Bearer {access_token}",
        "Accept": "application/json"
    }
    
    response = requests.get(f"{API_BASE}/users/{member_or_user_id}", headers=headers)
    
    if response.status_code == 200:
        user_info = response.json()
        print(f"✓ User info retrieved")
        return user_info
    else:
        print(f"⚠ Failed to get user info: {response.status_code}")
        return None


print("✓ Helper functions defined")

✓ Helper functions defined


## Task 2: Local Callback Server for Authorization Code Capture

This cell starts a temporary local HTTP server to capture the OAuth callback and extract the authorization code.

**Instructions:**
1. Run this cell - it will start a local server
2. Click the authorization URL that appears
3. Log in to Polar and authorize the application
4. The server will capture the authorization code automatically
5. The server will shut down after capturing the code

In [20]:
import secrets
import webbrowser
from http.server import BaseHTTPRequestHandler, HTTPServer
from urllib.parse import urlparse, parse_qs, urlencode
import threading

# Configuration
ALLOW_PORT_FALLBACK = os.getenv('ALLOW_PORT_FALLBACK', 'true').lower() == 'true'

# Global variables to capture authorization code
auth_code = None
auth_error = None
USED_REDIRECT_URI = None
state_token = secrets.token_urlsafe(32)


class CallbackHandler(BaseHTTPRequestHandler):
    """HTTP request handler for OAuth callback."""
    
    def log_message(self, format, *args):
        """Suppress default logging."""
        pass
    
    def do_GET(self):
        global auth_code, auth_error
        
        # Parse query parameters
        parsed_path = urlparse(self.path)
        query_params = parse_qs(parsed_path.query)
        
        # Validate state token
        received_state = query_params.get('state', [None])[0]
        if received_state != state_token:
            auth_error = "State token mismatch - possible CSRF attack"
            self.send_response(400)
            self.end_headers()
            self.wfile.write(b"Error: Invalid state token")
            return
        
        # Check for error
        if 'error' in query_params:
            auth_error = query_params.get('error_description', [query_params['error'][0]])[0]
            self.send_response(400)
            self.end_headers()
            self.wfile.write(f"Error: {auth_error}".encode())
            return
        
        # Extract authorization code
        if 'code' in query_params:
            auth_code = query_params['code'][0]
            self.send_response(200)
            self.send_header('Content-type', 'text/html')
            self.end_headers()
            html = """
            <html>
            <head><title>Authorization Successful</title></head>
            <body style="font-family: sans-serif; text-align: center; padding: 50px;">
                <h1 style="color: green;">✓ Authorization Successful!</h1>
                <p>You can close this window and return to the notebook.</p>
            </body>
            </html>
            """
            self.wfile.write(html.encode())
        else:
            auth_error = "No authorization code received"
            self.send_response(400)
            self.end_headers()
            self.wfile.write(b"Error: No authorization code")


def start_callback_server(port):
    """Start local callback server and return the actual port used."""
    global USED_REDIRECT_URI
    
    server = None
    actual_port = port
    
    # Try to bind to requested port
    try:
        server = HTTPServer(('localhost', port), CallbackHandler)
        actual_port = port
    except OSError as e:
        if not ALLOW_PORT_FALLBACK:
            raise RuntimeError(
                f"Port {port} is not available and ALLOW_PORT_FALLBACK is False. \n"
                f"Please free port {port} or set ALLOW_PORT_FALLBACK=true"
            ) from e
        
        # Try to find an available port
        print(f"⚠ Port {port} is busy, trying to find an available port...")
        for try_port in range(port + 1, port + 100):
            try:
                server = HTTPServer(('localhost', try_port), CallbackHandler)
                actual_port = try_port
                print(f"✓ Using fallback port: {actual_port}")
                break
            except OSError:
                continue
        
        if server is None:
            raise RuntimeError("Could not find an available port")
    
    USED_REDIRECT_URI = f"http://localhost:{actual_port}/callback"
    print(f"✓ Callback server ready at: {USED_REDIRECT_URI}")
    
    # Start server in background thread
    def serve():
        while auth_code is None and auth_error is None:
            server.handle_request()
        server.server_close()
    
    server_thread = threading.Thread(target=serve, daemon=True)
    server_thread.start()
    
    return actual_port, server_thread


# Start callback server
print("Starting local callback server...")
actual_port, server_thread = start_callback_server(REDIRECT_PORT)

# Build authorization URL
auth_params = {
    "response_type": "code",
    "client_id": CLIENT_ID,
    "redirect_uri": USED_REDIRECT_URI,
    "state": state_token
}
authorization_url = f"{AUTH_URL}?{urlencode(auth_params)}"

print("\n" + "="*80)
print("AUTHORIZATION REQUIRED")
print("="*80)
print(f"\n1. Click this URL to authorize:\n\n   {authorization_url}\n")
print("2. Log in to Polar and authorize this application")
print("3. You will be redirected back to localhost")
print("4. Wait for authorization code to be captured...\n")
print("="*80 + "\n")

# Wait for authorization code
server_thread.join(timeout=300)  # 5 minute timeout

if auth_error:
    print(f"\n❌ Authorization failed: {auth_error}")
    raise Exception(f"Authorization failed: {auth_error}")
elif auth_code:
    print(f"\n✓ Authorization code captured: {auth_code[:8]}...")
    print(f"✓ Redirect URI used: {USED_REDIRECT_URI}")
else:
    print("\n❌ Timeout waiting for authorization")
    raise Exception("Authorization timeout")

Starting local callback server...
⚠ Port 5000 is busy, trying to find an available port...
✓ Using fallback port: 5001
✓ Callback server ready at: http://localhost:5001/callback

AUTHORIZATION REQUIRED

1. Click this URL to authorize:

   https://flow.polar.com/oauth2/authorization?response_type=code&client_id=fddbcde4-65b8-4e25-8c74-08cab0a76e82&redirect_uri=http%3A%2F%2Flocalhost%3A5001%2Fcallback&state=kR12JWhvximgnqFJuQm81bIX-svdKkxgHmTKMM2IYXg

2. Log in to Polar and authorize this application
3. You will be redirected back to localhost
4. Wait for authorization code to be captured...



❌ Timeout waiting for authorization

❌ Timeout waiting for authorization


Exception: Authorization timeout

## Task 1b: Exchange Authorization Code for Tokens

Now that we have the authorization code, exchange it for access and refresh tokens.

In [ ]:
# Exchange authorization code for tokens
if not auth_code:
    raise Exception("No authorization code available. Please run Task 2 first.")

if not USED_REDIRECT_URI:
    raise Exception("USED_REDIRECT_URI not set. Please run Task 2 first.")

# Exchange code for tokens
token_response = exchange_code_for_token(auth_code, USED_REDIRECT_URI)

# Extract tokens
access_token = token_response.get('access_token')
refresh_token = token_response.get('refresh_token')
token_type = token_response.get('token_type', 'Bearer')
expires_in = token_response.get('expires_in')

# Save tokens to file
save_tokens(access_token, refresh_token, token_type)

# Display masked token info
print("\n" + "="*80)
print("TOKEN INFORMATION (masked)")
print("="*80)
print(f"Access Token:  {access_token[:8]}... (length: {len(access_token)})")
if refresh_token:
    print(f"Refresh Token: {refresh_token[:8]}... (length: {len(refresh_token)})")
print(f"Token Type:    {token_type}")
if expires_in:
    print(f"Expires In:    {expires_in} seconds ({expires_in//3600} hours)")
print("="*80 + "\n")

Exchanging authorization code for tokens...
  Using redirect_uri: http://localhost:5001/callback
✓ Token exchange successful
✓ Tokens saved to tokens_polar.json

TOKEN INFORMATION (masked)
Access Token:  ecce63b5... (length: 32)
Token Type:    bearer
Expires In:    315359999 seconds (87599 hours)



In [ ]:
# Optional: Demonstrate token refresh (run this when your access token expires)
# Uncomment to test:

# if refresh_token:
#     print("Demonstrating token refresh...")
#     new_token_response = refresh_access_token(refresh_token)
#     
#     # Update tokens
#     access_token = new_token_response.get('access_token')
#     new_refresh_token = new_token_response.get('refresh_token', refresh_token)
#     
#     # Save updated tokens
#     save_tokens(access_token, new_refresh_token, token_type)
#     
#     print(f"✓ New access token: {access_token[:8]}...")
#     if new_refresh_token != refresh_token:
#         print(f"✓ New refresh token: {new_refresh_token[:8]}...")
# else:
#     print("⚠ No refresh token available")

print("ℹ Token refresh demonstration cell (commented out by default)")

## Task 3: User Registration and Exercise Data Access

This section demonstrates:
1. Idempotent user registration (handles 409 conflicts)
2. Creating an exercise transaction
3. Handling 204 No Content responses gracefully
4. Listing and selecting exercises
5. Fetching heart rate zones
6. Committing the transaction

In [ ]:
# Load access token
tokens = load_tokens()
if not tokens:
    raise Exception("No tokens found. Please complete Tasks 1-2 first.")

access_token = tokens['access_token']

# User Registration (Idempotent)
print("Registering user...")

headers = {
    "Authorization": f"Bearer {access_token}",
    "Content-Type": "application/json",
    "Accept": "application/json"
}

# Prepare registration payload
registration_data = {}
if MEMBER_ID:
    registration_data["member-id"] = MEMBER_ID

response = requests.post(
    f"{API_BASE}/users",
    headers=headers,
    json=registration_data
)

polar_user_id = None

if response.status_code == 201:
    # Successfully registered
    user_data = response.json()
    polar_user_id = user_data.get('polar-user-id')
    print(f"✓ User registered successfully")
    print(f"  Polar User ID: {polar_user_id}")
    
elif response.status_code == 409:
    # User already registered
    print("ℹ User already registered (409 Conflict)")
    
    # Fetch user info to get polar-user-id
    user_id_to_fetch = MEMBER_ID if MEMBER_ID else "self"
    user_info = get_user_info(user_id_to_fetch, access_token)
    
    if user_info:
        polar_user_id = user_info.get('polar-user-id')
        print(f"✓ Retrieved Polar User ID: {polar_user_id}")
    else:
        print("⚠ Could not retrieve polar-user-id, will attempt to continue")
        
else:
    print(f"❌ User registration failed: {response.status_code}")
    print(f"   Response: {response.text}")
    raise Exception(f"User registration failed: {response.text}")

print(f"\n✓ User registration complete. Polar User ID: {polar_user_id or 'Unknown'}")

Registering user...
ℹ User already registered (409 Conflict)
Fetching user info for ID: 61732059...
✓ User info retrieved
✓ Retrieved Polar User ID: 61732059

✓ User registration complete. Polar User ID: 61732059


In [ ]:
# Listing user exercises using new (non-transaction) Polar AccessLink API and exporting latest TCX HR data
import pandas as pd
import xml.etree.ElementTree as ET
from io import BytesIO
from datetime import datetime

if not polar_user_id:
    raise Exception("Polar user ID unavailable. Ensure registration step completed.")

print("Listing exercises via /users/{user}/exercises API...\n")
exercises_url = f"{API_BASE}/exercises"
resp = requests.get(exercises_url, headers=headers)

exercises = []
if resp.status_code == 200:
    body = resp.json()
    if isinstance(body, list):
        exercises = body
    elif isinstance(body, dict):
        exercises = body.get('exercises', [])
    else:
        print(f"⚠ Unexpected exercises payload type: {type(body).__name__}")
    print(f"✓ Retrieved {len(exercises)} exercise(s)")
elif resp.status_code == 204:
    print("ℹ No exercises available (204 No Content)")
else:
    print(f"❌ Failed to list exercises: {resp.status_code}")
    print(f"   Response: {resp.text}")
    raise Exception("Exercise listing failed")

if not exercises:
    print("ℹ No exercises to process; skipping TCX download.")
else:
    print("\n" + "="*80)
    print("AVAILABLE EXERCISES (new API)")
    print("="*80)

    def get_field(ex, *keys):
        for key in keys:
            if key in ex:
                return ex[key]
        return None

    for i, ex in enumerate(exercises):
        print(f"\n{i+1}. Exercise ID: {get_field(ex, 'id', 'exercise_id')}")
        print(f"   Start Time: {get_field(ex, 'start_time', 'start-time', 'local_start_time')}")
        print(f"   Duration: {ex.get('duration', 'Unknown')}")
        print(f"   Sport: {get_field(ex, 'sport', 'detailed_sport_info')}")
    print("\n" + "="*80)

    def norm_start(ex):
        raw = get_field(ex, 'start_time', 'start-time', 'local_start_time', 'local-start-time')
        if not raw:
            return ''
        # Handle potential trailing Z
        raw_norm = raw.replace('Z', '+00:00') if raw.endswith('Z') else raw
        try:
            return datetime.fromisoformat(raw_norm)
        except ValueError:
            return raw

    latest = max(exercises, key=norm_start)
    exercise_id = get_field(latest, 'id', 'exercise_id')
    latest_start = get_field(latest, 'start_time', 'start-time', 'local_start_time')
    print(f"\n✓ Selected latest exercise: {exercise_id}")
    print(f"  Start Time: {latest_start}")

    # Fetch TCX for latest exercise
    print("\nDownloading TCX for latest exercise...")
    tcx_url = f"{API_BASE}/exercises/{exercise_id}/tcx"
    tcx_headers = {**headers, "Accept": "application/vnd.garmin.tcx+xml"}
    tcx_resp = requests.get(tcx_url, headers=tcx_headers)

    if tcx_resp.status_code != 200:
        print(f"❌ Failed to fetch TCX: {tcx_resp.status_code}")
        snippet = tcx_resp.text[:500] if hasattr(tcx_resp, 'text') else b""
        print(f"   Response: {snippet}...")
    else:
        print("✓ TCX downloaded; parsing XML...")
        try:
            tree = ET.parse(BytesIO(tcx_resp.content))
            root = tree.getroot()
        except ET.ParseError as e:
            print(f"❌ XML parse error: {e}")
            raise

        # TCX namespaces often present; handle default namespace
        ns = ''
        if root.tag.startswith('{'):
            ns = root.tag.split('}')[0].strip('{')
        def tag(name):
            return f"{{{ns}}}{name}" if ns else name

        trackpoints = []
        for activity in root.findall(f'.//{tag("Activity")}'):
            for lap in activity.findall(f'.//{tag("Lap")}'):
                for track in lap.findall(f'.//{tag("Track")}'):
                    for tp in track.findall(f'.//{tag("Trackpoint")}'):
                        time_elem = tp.find(tag("Time"))
                        hr_elem = tp.find(f'.//{tag("HeartRateBpm")}/{tag("Value")}')
                        t_val = time_elem.text if time_elem is not None else None
                        hr_val = hr_elem.text if hr_elem is not None else None
                        if hr_val is not None:
                            try:
                                hr_val = int(hr_val)
                            except ValueError:
                                pass
                        trackpoints.append({"Time": t_val, "HeartRateBpm": hr_val})

        if not trackpoints:
            print("⚠ No Trackpoint data found in TCX")
        else:
            df_tcx = pd.DataFrame(trackpoints)
            print(f"✓ Parsed {len(df_tcx)} trackpoints")
            csv_out_dir = Path("hr_data")
            csv_out_dir.mkdir(exist_ok=True)
            out_path = csv_out_dir / f"polar_latest_exercise_{exercise_id}.csv"
            df_tcx.to_csv(out_path, index=False)
            print(f"✓ Saved CSV: {out_path}")
            print("\nSample:")
            print(df_tcx.head(10))

print("\n" + "="*80)
print("EXERCISE FETCH (NEW API) COMPLETE")
print("="*80)

Listing exercises via /users/{user}/exercises API...

✓ Retrieved 51 exercise(s)

AVAILABLE EXERCISES (new API)

1. Exercise ID: 026aXeeA
   Start Time: 2025-09-18T10:49:02
   Duration: PT761S
   Sport: OTHER

2. Exercise ID: yBzEM6NJ
   Start Time: 2025-09-21T11:18:31
   Duration: PT1730S
   Sport: OTHER

3. Exercise ID: yV4RO9JZ
   Start Time: 2025-09-21T11:50:44
   Duration: PT465S
   Sport: OTHER

4. Exercise ID: 5p72Wqmq
   Start Time: 2025-09-22T10:29:35
   Duration: PT746S
   Sport: OTHER

5. Exercise ID: 5p72WgKO
   Start Time: 2025-09-22T10:45:05
   Duration: PT780S
   Sport: OTHER

6. Exercise ID: PngaWzAZ
   Start Time: 2025-09-22T11:00:34
   Duration: PT1555S
   Sport: OTHER

7. Exercise ID: y1BlOq6N
   Start Time: 2025-09-23T10:29:42
   Duration: PT1378S
   Sport: OTHER

8. Exercise ID: 0R926Wnj
   Start Time: 2025-09-23T10:57:07
   Duration: PT932S
   Sport: OTHER

9. Exercise ID: yMLqjGJG
   Start Time: 2025-09-24T11:34:36
   Duration: PT1395S
   Sport: OTHER

10. Exerci

## Task 4: Testing and Validation

### Testing Guidelines

#### Re-running Cells
- **Task 1**: Can be re-run safely to reload configuration
- **Task 2**: Should only be run once per authorization cycle (creates new auth code)
- **Task 1b**: Can be re-run to refresh tokens (requires valid auth_code variable)
- **Task 3**: Can be re-run to fetch latest exercises (may return 204 if no new data)

#### Handling 204 Responses
A 204 (No Content) response is **normal** and indicates:
- No new exercises available since last transaction
- No body content to parse (code handles this gracefully)

To trigger new data:
1. Sync a new workout from your Polar device to Polar Flow
2. Wait a few minutes for processing
3. Re-run Task 3

#### Port Fallback Behavior
- By default, if port 5000 is busy, the system finds an available port
- To disable fallback: `export ALLOW_PORT_FALLBACK=false`
- If fallback is disabled and port is busy, an error will be raised

#### Token Refresh
- Access tokens typically expire after a few hours
- Use the refresh token to get a new access token (see Task 1b)
- Refresh tokens are long-lived but may eventually expire

### Validation Checks

In [ ]:
# Validation: Check tokens file exists and has required fields
print("Running validation checks...\n")

validation_passed = True

# Check 1: Tokens file exists
if TOKENS_FILE.exists():
    print("✓ Tokens file exists")
    
    # Check 2: Tokens file is valid JSON
    try:
        tokens = load_tokens()
        print("✓ Tokens file is valid JSON")
        
        # Check 3: Required fields present
        required_fields = ['access_token', 'token_type']
        for field in required_fields:
            if field in tokens and tokens[field]:
                print(f"✓ Field '{field}' present")
            else:
                print(f"❌ Field '{field}' missing or empty")
                validation_passed = False
        
        # Check 4: Optional refresh_token
        if 'refresh_token' in tokens and tokens['refresh_token']:
            print(f"✓ Refresh token available")
        else:
            print(f"ℹ Refresh token not available (optional)")
        
        # Check 5: Token format (basic validation)
        if len(tokens['access_token']) > 10:
            print(f"✓ Access token format looks valid")
        else:
            print(f"⚠ Access token seems too short")
            validation_passed = False
            
    except json.JSONDecodeError:
        print("❌ Tokens file is not valid JSON")
        validation_passed = False
else:
    print("❌ Tokens file does not exist")
    print("   Run Tasks 1-2 to obtain tokens")
    validation_passed = False

# Check 6: Environment variables
env_vars = ['POLAR_CLIENT_ID', 'POLAR_CLIENT_SECRET']
for var in env_vars:
    if os.getenv(var):
        print(f"✓ Environment variable {var} is set")
    else:
        print(f"❌ Environment variable {var} is not set")
        validation_passed = False

# Summary
print("\n" + "="*80)
if validation_passed:
    print("✓ ALL VALIDATION CHECKS PASSED")
else:
    print("⚠ SOME VALIDATION CHECKS FAILED")
print("="*80)

## Troubleshooting

### Common Issues

#### 1. `invalid_request` or `unauthorized_client` errors
**Cause**: redirect_uri mismatch between authorization and token exchange

**Solution**: 
- Ensure the redirect URI registered in Polar developer portal matches exactly
- Check that USED_REDIRECT_URI variable is being passed to token exchange
- If using port fallback, register multiple redirect URIs or disable fallback

#### 2. JSONDecodeError on 204 responses
**Cause**: Attempting to parse empty response body

**Solution**: This notebook handles 204 responses correctly. If you encounter this:
- Ensure you're using the code from this notebook
- Check that you're not calling `.json()` on responses with status 204

#### 3. Port already in use
**Cause**: Another process is using the configured port

**Solution**:
- Allow fallback: `export ALLOW_PORT_FALLBACK=true` (default)
- Or free the port: `lsof -ti:5000 | xargs kill -9`
- Or use a different port: `export POLAR_REDIRECT_PORT=5001`

#### 4. 409 Conflict on user registration
**Cause**: User already registered

**Solution**: This is normal and handled gracefully. The code fetches the existing polar-user-id.

### Security Best Practices

1. **Never commit secrets**: Always use environment variables
2. **Rotate exposed credentials**: If you accidentally commit secrets, rotate them immediately
3. **Use HTTPS in production**: This notebook uses localhost for development only
4. **Validate state token**: Always implemented to prevent CSRF attacks
5. **Store tokens securely**: In production, use secure storage (keychain, secrets manager)

### Next Steps

- Explore other Polar API endpoints (activity, sleep, etc.)
- Integrate with your workout data analysis pipeline
- Automate token refresh in production applications
- Store exercise data in DuckDB (see populate_duckdb.ipynb)
